# 01 — Exploring the data and sensor quality

**Pygers-MEG preprocessing, step 1**

Before cleaning the data, look at it. This notebook gives you a first look at one recording and checks the
quality of every OPM sensor, so you know which sensors to trust before any filtering or artifact removal.

It follows two notebooks from the [FLUX pipeline for Cerca OPM systems](https://github.com/FLUX-pipeline/Cerca):
*A First Look at the Data* (`AFirstLook.ipynb`) and *Quality check of OPM sensors*
(`A2. Sensor_Quality_checking.ipynb`), adapted to our files.

**Part 1 — Data inspection (Steps 3–6)**
- What's in the recording: length, sampling rate, channels, filters (Step 3).
- How the channels are named and where the sensors sit on the head (Steps 4–5).
- The three measuring directions of each sensor, X, Y and Z (Step 6).

**Part 2 — Sensor quality (Steps 7–14)**
- A quick search for **flat** and **noisy** channels. Flat channels are marked bad; noisy ones are only
  listed, so you can look at them first (Step 7). Then a cell where you list the channels you want to mark
  bad yourself, with the reason for each (Step 7b), and the bad channels are saved (Step 7c).
- The **power spectrum** of every sensor, all directions and Z only (Step 8).
- The sensors **ranked by their average power**, as spectra and as a bar chart (Steps 9–10).
- The **time course** of the 3 noisiest and 3 quietest channels, and of any flat or noisy ones, over the whole
  recording and over 5 seconds (Steps 11–13).
- The **interactive data browser**, to scroll through the data yourself (Step 14).

### What you need before running this

- `05_Sync_eye_to_MEG.ipynb` (in `01-explore-raw-files/`) finished for this session. Like all the preprocessing
  notebooks, this one reads the synced MEG file in `derivatives/eyesync/` (named `…_proc-eyesync_…_meg.fif`).

### What it writes

The BIDS dataset and the synced file are only read. The one thing this notebook writes is the **list of bad
channels, with the reason for each** (Step 7c): a copy of the synced recording's `_channels.tsv` with its `status` and
`status_description` columns filled in, in a new derivatives folder:

| Folder | Files |
|---|---|
| `derivatives/sensorqc/sub-<label>/ses-<label>/meg/` | `…_proc-eyesync_channels.tsv` (one row per channel; `status` is `good` or `bad`, and `status_description` says why a channel is bad) and `…_channels.json` (what the columns mean) |

Later preprocessing notebooks read the bad channels from there.

## Step 0: Load the Python packages

This cell loads the tools the notebook uses. You don't need to change anything.

In [ ]:
# --- Standard Python tools (come with Python) ---
from pathlib import Path        # build and check file/folder paths
import re                       # "regular expressions": find patterns in text (used to split channel names)

# --- Scientific Python tools ---
import numpy as np              # numbers and arrays (lists of numbers you can do math on)
import pandas as pd             # tables
import matplotlib.pyplot as plt # plotting
from matplotlib.patches import Patch   # coloured squares for a plot legend
from scipy.signal import detrend       # removes a straight-line trend from a signal (Steps 12-13)
from IPython.display import display   # shows a table nicely in the notebook

# --- MNE-Python: the MEG analysis package ---
import mne

# --- MNE-BIDS: reads BIDS files together with their sidecar files (Step 2) ---
import mne_bids

# Only show MNE's warnings and errors (not every info message)
mne.set_log_level("WARNING")

# Draw plots inside the notebook (Step 14 switches to an interactive window at the very end)
%matplotlib inline

print("MNE version:", mne.__version__)
print("MNE-BIDS version:", mne_bids.__version__)

## Step 1: Settings ✏️ (edit this cell)

Everything you normally need to change is in the cell below. The only other cell you edit is Step 7b, where
you list the channels you want to mark bad.

In [ ]:
# ---------------------------------------------------------------------------
# WHICH RECORDING
# ---------------------------------------------------------------------------
BIDS_ROOT = Path("/jukebox/[yourpath]/auditory-oddball/bids")  # top folder of your BIDS dataset
SUBJECT = "01"           # the part after "sub-"  (as text, in quotes)
SESSION = "01"           # the part after "ses-"; use None (no quotes) if you have no sessions
TASK = "oddballftcued"   # the part after "task-"
RUN = None               # the part after "run-", e.g. "01"; use None if there is no run label
EYESYNC_DERIV = "eyesync"    # the derivatives folder notebook 05 wrote the synced MEG file to
SENSORQC_DERIV = "sensorqc"  # the derivatives folder this notebook writes the bad-channel list to (Step 7c)

# ---------------------------------------------------------------------------
# SEARCH FOR FLAT AND NOISY CHANNELS (Step 7; the same screen as 01_Explore_Raw_OPM-MEG_Files.ipynb, Section 6.1)
# ---------------------------------------------------------------------------
SCREEN_TMIN = 30.0   # start of the screened stretch, in seconds from the start of the recording
SCREEN_DUR = 60.0    # length of the screened stretch, in seconds
SCREEN_Z = 5.0       # robust z-score above which a channel is "noisy" (below minus this: "flat")
FLAT_MARKS_WHOLE_SENSOR = True   # True: a flat channel marks all three directions (X, Y, Z) of its sensor
                                 # bad; False: only the flat channel itself

# ---------------------------------------------------------------------------
# POWER SPECTRA (Steps 8-10)
# ---------------------------------------------------------------------------
PSD_FMIN = 1.0             # lowest frequency (Hz). Starting at 1 Hz leaves out the slow drift (see Step 8)
PSD_FMAX = 120.0           # highest frequency (Hz)
PSD_RESOLUTION_HZ = 0.5    # spacing between frequencies in the spectrum (Hz)

# ---------------------------------------------------------------------------
# TIME COURSES (Steps 11-13)
# ---------------------------------------------------------------------------
N_RANKED = 3               # how many of the noisiest and of the quietest channels to plot
MAX_FLAGGED_PLOT = 12      # plot at most this many of the channels flagged in Step 7 (the most extreme first)
STRETCH_START_S = None     # start of the short stretch (s); None: the middle of the recording
STRETCH_DUR_S = 5.0        # length of the short stretch (s)

# ---------------------------------------------------------------------------
# INTERACTIVE DATA BROWSER (Step 14)
# ---------------------------------------------------------------------------
BROWSE_START_S = 60.0      # where the browser starts (s)
BROWSE_DURATION_S = 10.0   # how many seconds it shows at a time
BROWSE_HIGHPASS_HZ = 1.0   # high-pass filter for the DISPLAY only (Hz), so slow drift doesn't hide everything
                           # else; the data are not changed. None: show the data unfiltered

## Step 2: Find the file and load the data

This cell works out the path of the synced MEG file from your settings, checks it exists, and reads it.
If something is wrong it stops with a message that says what to fix.

`preload=True` reads all the data into memory. This notebook needs that, because it computes spectra over
the whole recording. A long recording can take a minute or two to load.

In [ ]:
BIDS_ROOT = Path(BIDS_ROOT)

if "[yourpath]" in str(BIDS_ROOT):
    raise ValueError("BIDS_ROOT still contains the [yourpath] placeholder. Set it in Step 1.")

# ---- The BIDS-style name of this recording, e.g. sub-01_ses-01_task-oddball ------------------
sub_ses = [f"sub-{SUBJECT}"] + ([f"ses-{SESSION}"] if SESSION else [])   # folder levels
name_parts = sub_ses + [f"task-{TASK}"] + ([f"run-{RUN}"] if RUN else [])
base = "_".join(name_parts)        # the start of every file name for this recording
sub_dir = Path(*sub_ses)           # e.g. sub-01/ses-01

# ---- The synced MEG file from notebook 05 (MEG + eye channels) --------------------------------
eyesync_root = BIDS_ROOT / "derivatives" / EYESYNC_DERIV
# BIDSPath describes a file by its BIDS labels. Notebook 05 labels the synced file "proc-eyesync"
# (proc: a processed version of this recording).
bids_path = mne_bids.BIDSPath(root=eyesync_root, subject=SUBJECT, session=SESSION, task=TASK, run=RUN,
                              processing="eyesync", datatype="meg", suffix="meg", extension=".fif")
# FIF files can't be bigger than 2 GB, so MNE saves a longer recording in parts:
# ..._split-01_meg.fif, ..._split-02_meg.fif, ... We open the first part and MNE finds the others itself.
if not bids_path.fpath.is_file() and bids_path.copy().update(split="01").fpath.is_file():
    bids_path.update(split="01")
meg_file = bids_path.fpath
meg_dir = meg_file.parent

if not meg_file.is_file():
    found = sorted(meg_dir.glob("*_meg.fif")) if meg_dir.is_dir() else []
    raise FileNotFoundError(f"Synced MEG file not found:\n    {meg_file}\nHas notebook 05 been run for this "
                            "session? FIF files in that folder: " + (", ".join(p.name for p in found) or "none"))

# ---- The synced recording's _channels.tsv (written by notebook 05, next to the FIF file) -----------
# It lists every channel, the eye-tracking ones included, with any channels already marked bad and why.
# Step 7c saves a copy of it with the bad channels filled in.
bids_channels_tsv = bids_path.copy().update(split=None, suffix="channels", extension=".tsv").fpath
if not bids_channels_tsv.is_file():
    raise FileNotFoundError(f"The synced recording's _channels.tsv was not found:\n    {bids_channels_tsv}\n"
                            "Re-run notebook 05 for this session; newer versions write it.")

# ---- Read it --------------------------------------------------------------------------------
# read_raw_bids reads the FIF file together with its sidecar files in derivatives/eyesync/: the channel types
# and bad channels come from _channels.tsv, and the task and eye-tracking events from _events.tsv (they appear
# as annotations).
#
# You will see one warning here, "participants.tsv file not found". It is expected and harmless:
# read_raw_bids also looks for participants.tsv (age, sex, handedness, ...) in the top folder of the dataset
# it reads from, to copy those details into the recording's header. Here that folder is derivatives/eyesync/,
# but participants.tsv only exists at the top of the main BIDS dataset, so it isn't found and those details
# are not added. Nothing in this notebook uses them.
raw = mne_bids.read_raw_bids(bids_path)
raw.load_data()   # read the data themselves into memory (the steps below need them)

# ---- Repair the three eye-tracker channels (ET_xpos, ET_ypos, ET_pupil) ------------------------------
# What goes wrong: in the header, every channel has a "kind" (MEG sensor, trigger, eye tracker, ...) and a
# more detailed "coil type". For eye-tracker channels MNE uses the coil type to tell gaze position (ET_xpos,
# ET_ypos) apart from pupil size (ET_pupil). When read_raw_bids applies the channel types listed in
# _channels.tsv (EYEGAZE, PUPIL), it sets the kind correctly but resets the coil type of these three channels
# to "none". MNE then can't work out what they are, and even simple calls like raw.get_channel_types() stop
# with an error. This is a limitation of MNE-BIDS (version 0.20) with eye-tracking channels, not a problem
# with your files.
#
# The fix: the FIF file's own header still has the right coil types. We read just the header (quick; no
# data) and, for every channel whose kind is the same but whose coil type was changed, copy the coil type
# back. Only the three eye-tracker channels are affected; everything else stays exactly as read_raw_bids
# loaded it.
header = mne.io.read_info(meg_file, verbose=False)
repaired = []
for ch_in_file in header["chs"]:
    if ch_in_file["ch_name"] in raw.ch_names:
        ch = raw.info["chs"][raw.ch_names.index(ch_in_file["ch_name"])]
        if ch["kind"] == ch_in_file["kind"] and ch["coil_type"] != ch_in_file["coil_type"]:
            ch["coil_type"] = ch_in_file["coil_type"]
            repaired.append(ch["ch_name"])
if repaired:
    print("Repaired the channel description of:", ", ".join(repaired))

print("Loaded:", meg_file)
if len(raw.filenames) > 1:
    print(f"(the recording is split into {len(raw.filenames)} files; MNE read them all)")

# Part 1: Data inspection

## Step 3: A first look at the recording

Printing `raw` gives a one-line summary: the file name, how many channels, how long the recording is and how
much memory it takes.

In [ ]:
print(raw)

`raw.info` is the file's **header**: everything about the recording except the data itself. Printing it
lists the main fields.

In [ ]:
print(raw.info)

The fields worth knowing about:

| Field | What it is |
|---|---|
| `sfreq` | The **sampling rate**: how many samples per second each channel has. |
| `lowpass` / `highpass` | The filters already applied to the data. In a raw recording these are the hardware's limits; every filter you apply later updates them. |
| `line_freq` | The **mains (power line) frequency**: 60 Hz in North America (cMEG2FIF writes it), 50 Hz in Europe and much of Asia. Line noise shows up at this frequency and its multiples (120, 180 Hz, ...). |
| `bads` | The channels marked bad. Empty unless someone marked channels during the recording. Step 7 adds to it. |
| `chs` | One entry per channel: its name, type, position and measuring direction. |
| `dig` | The digitized points: the fiducials and the EinScan head shape (used for coregistration, notebook TBD2). |
| `dev_head_t` | The **device → head transform**: where the sensor helmet sits relative to the head. |
| `meas_date` | When the recording was made (shifted for anonymization by notebook 04). |

## Step 4: The channel names

The recording has several **kinds** of channels. This cell counts them, then prints the names of the first 40.

In [ ]:
# Count the channels of each type, e.g. mag: 192, ref_meg: 6, stim: 20, misc: ...
types = pd.Series(raw.get_channel_types()).value_counts()
print("Channels of each type:")
print(types.to_string())

# The names of the first 40 channels. raw.info["ch_names"] is a list, and [:40] takes its first 40 items.
ch_names = raw.info["ch_names"]
print("\nThe first 40 channel names:")
print(ch_names[:40])

What the channel types are:

- **`mag`**: the OPM sensors on the head. These are the brain data.
- **`ref_meg`**: reference sensors, away from the head, that record only the background field.
- **`stim`**: trigger channels: the codes from the stimulus computer (`STI101`), the button presses (`STI_BTN`)
  and the raw trigger lines.
- **`eyegaze`** and **`pupil`**: the EyeLink gaze position (`ET_xpos`, `ET_ypos`) and pupil size (`ET_pupil`)
  that notebook 05 added.
- **`misc`**: everything else: the analog inputs (including the EyeLink's analog outputs `eye_x`, `eye_y` and
  `eye_pupil`) and the other eye-tracking channels from notebook 05.

**How an OPM channel is named.** Each OPM sensor measures the magnetic field in **three directions**, so it
appears as **three channels**. A name like `2C[X]` means "sensor `2C`, X direction"; the same sensor's other
two channels are `2C[Y]` and `2C[Z]`.

## Step 5: Where the sensors are

This draws the sensor positions seen from above (nose at the top), the way MNE shows a topographic map.

Each sensor position holds three channels (X, Y and Z) at the same place, so we draw only the Z channel of each
sensor. That gives one dot and one readable label per sensor. Channels already marked bad are drawn in red.

In [ ]:
# All Z-direction OPM channels (Step 6 explains the directions)
z_names = [name for name in raw.ch_names if name.endswith("[Z]")]

# The header ("info") for just those channels, bad ones included. Only the header is copied, not the data.
info_z = mne.pick_info(raw.info, mne.pick_channels(raw.ch_names, include=z_names))

# kind="topomap": seen from above. show_names=True writes each channel's name next to its dot.
# The ";" at the end stops Jupyter from also printing the figure object as text.
mne.viz.plot_sensors(info_z, kind="topomap", show_names=True, title="OPM sensor positions (Z channels)");

## Step 6: The three measuring directions (X, Y and Z)

Each sensor measures the field in three directions:

- **Z** points **out of the head**, at right angles to the scalp. This is what a conventional (SQUID) MEG
  magnetometer measures, so most MEG methods and pictures are made for it. The field from a brain source changes
  most clearly in this direction, which makes it the easiest to interpret.
- **X** and **Y** lie **along the scalp**, at right angles to each other and to Z.

This cell makes three lists of channel names, one per direction, which the rest of the notebook uses. It looks
only at the OPM channels on the head (type `mag`): the reference sensors and the other channels are left out,
even if a name happened to end in the same letter.

In [ ]:
# The names of the OPM channels on the head (type "mag"), including any already marked bad
opm_names = [raw.ch_names[i] for i in mne.pick_types(raw.info, meg="mag", ref_meg=False, exclude=[])]

# A channel name has the form "<sensor>[<direction>]", e.g. "2C[X]". This pattern splits it into its two parts:
#   ^(.*?)       the sensor name (everything before the bracket)
#   \s*          any spaces before the bracket (none in our files; allowed in case another converter adds one)
#   \[([XYZ])\]$  the direction: X, Y or Z inside square brackets, at the very end
NAME_PATTERN = re.compile(r"^(.*?)\s*\[([XYZ])\]$")


def sensor_of(name):
    # The sensor a channel belongs to, e.g. "2C[X]" -> "2C" (the name itself if it doesn't fit the pattern)
    found = NAME_PATTERN.match(name)
    return found.group(1) if found else name


def direction_of(name):
    # The direction a channel measures, e.g. "2C[X]" -> "X" ("?" if the name doesn't fit the pattern)
    found = NAME_PATTERN.match(name)
    return found.group(2) if found else "?"


OPMX = [name for name in opm_names if direction_of(name) == "X"]
OPMY = [name for name in opm_names if direction_of(name) == "Y"]
OPMZ = [name for name in opm_names if direction_of(name) == "Z"]
odd = [name for name in opm_names if direction_of(name) == "?"]

print(f"OPM channels: {len(opm_names)} on {len({sensor_of(n) for n in opm_names})} sensors")
print(f"  X direction: {len(OPMX)}")
print(f"  Y direction: {len(OPMY)}")
print(f"  Z direction: {len(OPMZ)}")
if odd:
    print("  Names that don't end in [X], [Y] or [Z] (left out of the three lists):", odd)
print("\nThe first five Z channels:", OPMZ[:5])

# Part 2: Sensor quality

## Step 7: Search for flat and noisy channels

The same quick screen as `01_Explore_Raw_OPM-MEG_Files.ipynb` (in `01-explore-raw-files/`, Section 6.1), now on the synced file. On one stretch of data
(`SCREEN_DUR` seconds starting at `SCREEN_TMIN`), filtered to 1–40 Hz so that slow drift and line noise don't
count, it measures how much each OPM channel's signal varies (its **standard deviation**) and compares it with
all the others:

- A channel that varies **far less** than the rest is **flat**: the sensor is off, not locked, or not
  recording. A flat channel holds no brain signal, so it is **marked bad** now. With
  `FLAT_MARKS_WHOLE_SENSOR = True`, the other two directions of the same sensor are marked bad too, since
  a problem like this usually affects the whole sensor.
- A channel that varies **far more** than the rest is **noisy**. Noisy channels are **not** marked bad
  here. They are only listed, because "noisy" can have several causes (a loose sensor, a muscle or
  movement artifact on that part of the head, an interference source nearby), and some can be fixed by later
  cleaning steps. Steps 11–13 plot them so you can decide.

**How "far" is measured.** The screen works with the logarithm of each channel's standard deviation and its
**robust z-score**: how many "typical spreads" a channel is from the median channel. It uses the median and the
median absolute deviation instead of the mean and standard deviation, so a few very bad channels can't hide
themselves by pulling the average towards them. A channel more than `SCREEN_Z` (5) spreads above the median is
noisy, more than 5 below it is flat.

In [ ]:
def screen_channels(r, names, tmin=30.0, dur=60.0, z_thresh=5.0):
    # The flat/noisy screen from 01_Explore_Raw_OPM-MEG_Files.ipynb: a table with one row per channel in `names`.
    tmax = min(tmin + dur, r.times[-1])          # don't run past the end of the recording
    tmin = max(0.0, tmax - dur)                  # (a short recording: use its last `dur` seconds)
    data = r.get_data(picks=names, tmin=tmin, tmax=tmax)   # just that stretch and those channels (tesla)
    data = mne.filter.filter_data(data, r.info["sfreq"], 1.0, 40.0, verbose=False)   # ignore drift and line noise
    sd = data.std(axis=1)                        # standard deviation of each channel (tesla)
    log_sd = np.log10(sd + 1e-30)                # log scale (the tiny 1e-30 avoids log of 0 for dead channels)
    med = np.median(log_sd)
    mad = np.median(np.abs(log_sd - med)) * 1.4826 + 1e-12   # robust "typical spread"
    z = (log_sd - med) / mad                     # robust z-score
    table = pd.DataFrame({"channel": list(names), "sd_fT": sd * 1e15, "robust_z": z})
    table["flag"] = np.where(z > z_thresh, "noisy", np.where(z < -z_thresh, "flat", ""))
    return table.sort_values("robust_z", key=np.abs, ascending=False)   # most extreme first


# ---- The channels already bad in the file, and why ---------------------------------------------
# bad_reasons holds the reason for every bad channel (channel name -> reason). Step 7c writes it to the
# status_description column of _channels.tsv.
bads_at_start = list(raw.info["bads"])           # bad channels already marked (from _channels.tsv)
bids_channels = pd.read_csv(bids_channels_tsv, sep="\t", dtype=str, keep_default_na=False)   # all as text
earlier_reasons = dict(zip(bids_channels["name"], bids_channels.get("status_description", "")))
bad_reasons = {}
for name in bads_at_start:
    reason = earlier_reasons.get(name, "")
    bad_reasons[name] = reason if reason not in ("", "n/a") else "marked bad before this notebook (no reason given)"
print("Already marked bad in the file:", bads_at_start or "none")

screen = screen_channels(raw, opm_names, SCREEN_TMIN, SCREEN_DUR, SCREEN_Z)
flat_channels = list(screen.loc[screen["flag"] == "flat", "channel"])
noisy_channels = list(screen.loc[screen["flag"] == "noisy", "channel"])

print(f"\nScreened {SCREEN_DUR:g} s from {SCREEN_TMIN:g} s, 1-40 Hz, robust z beyond ±{SCREEN_Z:g}:")
print(f"  flat : {flat_channels or 'none'}")
print(f"  noisy: {noisy_channels or 'none'}")
print("\nThe most extreme channels (flagged ones, or the top 10 if none are flagged):")
flagged = screen[screen["flag"] != ""]
display(flagged if len(flagged) else screen.head(10))

# ---- Mark the flat channels bad, each with its reason -------------------------------------------
z_of = dict(zip(screen["channel"], screen["robust_z"]))   # each channel's robust z-score
screen_note = f"{SCREEN_DUR:g} s from {SCREEN_TMIN:g} s, 1-40 Hz"
newly_bad = []
for name in flat_channels:
    newly_bad.append(name)
    bad_reasons.setdefault(name, f"flat signal (sensor-quality screen: robust z = {z_of[name]:+.1f}; {screen_note})")
if FLAT_MARKS_WHOLE_SENSOR:
    for name in opm_names:   # the other directions of each flat sensor
        if sensor_of(name) in {sensor_of(f) for f in flat_channels} and name not in newly_bad:
            newly_bad.append(name)
            bad_reasons.setdefault(name, f"another direction of sensor {sensor_of(name)} is flat (sensor-quality screen)")
raw.info["bads"] = sorted(set(raw.info["bads"]) | set(newly_bad))   # add them, without repeats

# Remember this state, so Step 7b can start again from here each time you re-run it
bads_after_screen = list(raw.info["bads"])
reasons_after_screen = dict(bad_reasons)

print("\nMarked bad now:", [n for n in newly_bad if n not in bads_at_start] or "nothing new")
print("All bad channels:", raw.info["bads"] or "none")
if noisy_channels:
    print(f"\n{len(noisy_channels)} noisy channel(s) NOT marked bad; they are plotted in Steps 11 and 13.")

> **The screen only suggests.** It looks at one minute, with fixed limits. Always confirm by looking at the
> spectra (Steps 8–10) and the traces (Steps 11–14) before you decide. Step 7b is where you mark channels
> bad yourself.

## Step 7b: Mark channels bad by hand ✏️ (edit this cell)

List the channels you want to mark bad in `BAD_CHANNELS` below. You can write:

- a **channel name**, e.g. `"5F[Y]"`, to mark that one channel, or
- a **sensor name**, e.g. `"2C"`, to mark all three of its channels (`2C[X]`, `2C[Y]` and `2C[Z]`).
  On OPMs a problem usually affects the whole sensor, so this is often what you want.

Names must be written exactly as in the file, inside quotes, separated by commas. Leave the list empty (`[]`)
to mark nothing.

In `BAD_REASONS`, say **why** each one is bad, using the same name as in `BAD_CHANNELS` (the same two settings as
notebook 04). The reason is saved with the channel in Step 7c, so anyone using the data later can see why it was
left out. A reason is optional but strongly recommended: a channel without one is saved as "marked bad by hand
(no reason given)".

**How to use it:** the first time, run the whole notebook with the list empty. Look at the spectra, the ranking
and the traces (Steps 8–14). Then come back here, add the channels you decided are bad, and run this cell and
**every cell after it** again (in Jupyter: click this cell, then *Run → Run Selected Cell and All Below*). The
spectra, ranking and plots are then made without them.

The channels are added to the ones already marked bad (from the file and from Step 7). Each time you run this
cell it starts again from the state after Step 7, so taking a channel **out** of the list and re-running also
unmarks it. Channels marked bad earlier (by notebook 04, or flat in Step 7) can't be unmarked here.

In [ ]:
# ---------------------------------------------------------------------------
# YOUR BAD CHANNELS: channel names ("5F[Y]") or sensor names ("2C") in quotes, separated by commas
# ---------------------------------------------------------------------------
BAD_CHANNELS = []         # e.g. ["2C", "5F[Y]"]
BAD_REASONS = {}          # why each one is bad, e.g. {"2C": "flat signal", "5F[Y]": "noisy, bursts of high-frequency noise"}
# ---------------------------------------------------------------------------

# Start again from the state after Step 7, so re-running this cell with a shorter list also unmarks channels
raw.info["bads"] = list(bads_after_screen)
bad_reasons = dict(reasons_after_screen)

# A reason for something not in BAD_CHANNELS is almost always a typo
stray = [key for key in BAD_REASONS if key not in BAD_CHANNELS]
if stray:
    raise ValueError(f"BAD_REASONS has a reason for {stray}, which is not in BAD_CHANNELS. "
                     "Use exactly the same name in both.")

sensor_names = {sensor_of(name) for name in opm_names}   # every OPM sensor, e.g. "2C"
to_mark, unknown = [], []
for entry in BAD_CHANNELS:
    if entry in raw.ch_names:                        # a channel name: mark that channel
        channels = [entry]
    elif entry in sensor_names:                      # a sensor name: mark all its channels (X, Y and Z)
        channels = [name for name in opm_names if sensor_of(name) == entry]
    else:                                            # neither: probably a typo
        unknown.append(entry)
        continue
    reason = BAD_REASONS.get(entry, "").strip() or "marked bad by hand (no reason given)"
    for name in channels:
        to_mark.append(name)
        bad_reasons[name] = reason                   # your reason replaces an earlier one

if unknown:
    raise ValueError(f"Not a channel or sensor name in this recording: {unknown}\n"
                     "Check the spelling, including the square brackets, e.g. \"5F[Y]\". "
                     "Step 4 prints the channel names.")

new = [name for name in to_mark if name not in raw.info["bads"]]
raw.info["bads"] = sorted(set(raw.info["bads"]) | set(to_mark))   # add them, without repeats

print("Marked bad here:", new or "nothing new")
print(f"\nAll bad channels ({len(raw.info['bads'])}), with the reason saved in Step 7c:")
display(pd.DataFrame({"channel": raw.info["bads"],
                      "reason": [bad_reasons.get(n, "") for n in raw.info["bads"]]}))

## Step 7c: Save the bad channels

This writes the bad channels and their reasons to `derivatives/sensorqc/sub-<label>/ses-<label>/meg/`, as a
**copy of the synced recording's `_channels.tsv`** from `derivatives/eyesync/` (the file notebook 05 wrote, which
also lists the eye-tracking channels; the original is not changed). In the copy, every channel's **`status`** is `good` or `bad`, and **`status_description`** gives
the reason for each bad one. `_channels.json`, which describes the columns, is copied next to it. The folder
also gets a `dataset_description.json` the first time, like every derivatives folder.

**Re-run this cell whenever you change the bad channels in Step 7b**, so the saved list matches. (*Run Selected
Cell and All Below* from Step 7b does this.) Channels you mark by clicking in the data browser (Step 14) are
**not** saved: add them to `BAD_CHANNELS` in Step 7b and re-run from there.

To use the list in a later notebook:

```python
channels = pd.read_csv(".../derivatives/sensorqc/sub-01/ses-01/meg/sub-01_ses-01_task-…_proc-eyesync_channels.tsv", sep="\t")
raw.info["bads"] = list(channels.loc[channels["status"] == "bad", "name"])
```

In [ ]:
import json        # reads and writes .json files

# ---- Where it goes ----------------------------------------------------------------------------
sensorqc_root = BIDS_ROOT / "derivatives" / SENSORQC_DERIV
sensorqc_dir = sensorqc_root / sub_dir / "meg"
# Same name as the file it copies (…_proc-eyesync_channels.tsv), so the name says which recording it describes:
# the eye-synced one, which also has the eye-tracking channels.
sensorqc_tsv = sensorqc_dir / bids_channels_tsv.name

# ---- dataset_description.json for the derivatives folder (made once; its BIDS version matches the data's) --
raw_version = json.loads((BIDS_ROOT / "dataset_description.json").read_text()).get("BIDSVersion")
description_file = sensorqc_root / "dataset_description.json"
if description_file.exists():
    description = json.loads(description_file.read_text())   # made by an earlier run: keep it
    description["BIDSVersion"] = raw_version
else:
    description = {"Name": "OPM sensor quality: bad channels", "BIDSVersion": raw_version,
                   "DatasetType": "derivative",
                   "GeneratedBy": [{"Name": "01_Explore_Data_and_Sensor_Quality.ipynb (Pygers-MEG)",
                                    "Description": "Bad channels from a flat/noisy screen and inspection of "
                                                   f"spectra and traces, MNE-Python {mne.__version__}"}]}
# DatasetLinks: where the datasets this one refers to are, relative to this folder. "raw" is the main BIDS
# dataset; "eyesync" holds the recording these bad channels belong to (see "Sources" in _channels.json below).
description.setdefault("DatasetLinks", {}).update({"raw": "../../", "eyesync": f"../{EYESYNC_DERIV}/"})
sensorqc_dir.mkdir(parents=True, exist_ok=True)
description_file.write_text(json.dumps(description, indent=4))

# ---- The _channels.tsv copy with status and status_description filled in -----------------------
channels = bids_channels.copy()                     # read in Step 7, all columns as text
was_bad = channels.get("status", pd.Series("", index=channels.index)) == "bad"
is_bad = channels["name"].isin(raw.info["bads"])
channels["status"] = np.where(is_bad, "bad", "good")
if "status_description" not in channels:
    channels["status_description"] = "n/a"
# Bad channels: their reason. Good channels: keep any note they had, unless it was a reason for being bad.
channels.loc[is_bad, "status_description"] = [bad_reasons.get(n, "n/a") for n in channels.loc[is_bad, "name"]]
channels.loc[~is_bad & was_bad, "status_description"] = "n/a"
channels.to_csv(sensorqc_tsv, sep="\t", index=False)

# _channels.json describes the columns. Copy notebook 05's, and add "Sources": the recording file(s) this list
# of channels belongs to. "bids:eyesync:" means "inside the eyesync folder", as named in DatasetLinks above.
bids_channels_json = bids_channels_tsv.with_suffix(".json")
channels_meta = json.loads(bids_channels_json.read_text()) if bids_channels_json.is_file() else {}
channels_meta["Sources"] = [f"bids:eyesync:{(sub_dir / 'meg' / Path(f).name).as_posix()}" for f in raw.filenames]
sensorqc_tsv.with_suffix(".json").write_text(json.dumps(channels_meta, indent=4))

# ---- Report ------------------------------------------------------------------------------------
print("Saved:", sensorqc_tsv)
not_in_tsv = [n for n in raw.info["bads"] if n not in set(channels["name"])]
if not_in_tsv:
    print("Bad channels that are not in _channels.tsv, so not saved:",
          not_in_tsv)
print(f"\n{int(is_bad.sum())} bad channel(s) in the saved file:")
display(channels.loc[is_bad, ["name", "type", "status", "status_description"]].reset_index(drop=True))

## Step 8: The power spectra

A **power spectrum** (power spectral density, PSD) shows how much of a signal's power is at each frequency.
It is the quickest way to see a sensor's noise: a good sensor has a smooth spectrum, falling with frequency,
with sharp peaks at the line frequency (60 Hz) and its multiples, and perhaps a bump around 8–12 Hz (alpha
rhythm). A broken or loose sensor stands out as much higher (or lower) than the others.

How it's computed (**Welch's method**): the recording is cut into overlapping windows (each half-overlapping the
next), the spectrum of each window is computed, and the spectra are averaged. The window length sets the
**frequency resolution**: a window of 2 s gives 0.5 Hz steps (`PSD_RESOLUTION_HZ`). In MNE the window length is
given as a number of samples, `n_fft`.

The spectrum starts at `PSD_FMIN` (1 Hz). OPMs are DC-coupled and record slow drifts of the background field;
including 0 Hz would let that drift dominate the picture and the ranking in Step 9.

Bad channels are left out (`exclude="bads"`), so the flat ones from Step 7 don't stretch the scale.

In [ ]:
sfreq = raw.info["sfreq"]
n_fft = int(round(sfreq / PSD_RESOLUTION_HZ))   # window length in samples: e.g. 750 Hz / 0.5 Hz = 1500
fmax = min(PSD_FMAX, sfreq / 2)                 # can't go above half the sampling rate (the Nyquist frequency)
print(f"Sampling rate {sfreq:g} Hz -> n_fft = {n_fft} samples ({n_fft / sfreq:g} s windows, "
      f"{sfreq / n_fft:g} Hz resolution), {PSD_FMIN:g}-{fmax:g} Hz")

# The OPM channels that are not marked bad. Passing this list (rather than picks="mag") makes sure the
# spectra are computed for exactly these channels and no others.
good_opm = [name for name in opm_names if name not in raw.info["bads"]]
psd_all = raw.compute_psd(method="welch", fmin=PSD_FMIN, fmax=fmax, picks=good_opm,
                          n_fft=n_fft, n_overlap=n_fft // 2)

# One line per channel; MNE converts the power to decibels (dB) for the plot
fig = psd_all.plot(exclude="bads", show=False)
fig.suptitle("Power spectra: all OPM channels (X, Y and Z), bad channels left out")
plt.show()

And the **Z direction only**, the direction that is closest to a conventional MEG magnetometer (Step 6).

In [ ]:
z_good = [name for name in OPMZ if name not in raw.info["bads"]]   # Z channels that are not bad
psd_z = raw.compute_psd(method="welch", fmin=PSD_FMIN, fmax=fmax, picks=z_good,
                        n_fft=n_fft, n_overlap=n_fft // 2)

fig = psd_z.plot(exclude="bads", show=False)
fig.suptitle("Power spectra: Z direction only, bad channels left out")
plt.show()

What to look for:

- **Peaks at 60, 120 and 180 Hz**: line noise from the mains (60 Hz in North America; it would be 50 Hz in
  Europe). Every channel has some. It is removed later.
- **A bump around 8–12 Hz** in some channels: the alpha rhythm, strongest over the back of the head with
  the eyes closed or relaxed.
- **Lines that sit well above the rest**: noisy channels. **Lines well below**: channels that are
  partly dead. Steps 9–10 put a name to each line.
- **The X and Y directions are often noisier** than Z, because they pick up more of the room's interference,
  so the "all directions" plot usually spreads wider than the Z-only one.

## Step 9: Rank the sensors by their average power

To see which sensors are noisiest, each channel's spectrum is converted to **decibels** and averaged over all
frequencies (`PSD_FMIN`–`PSD_FMAX`). That one number per channel is its **average power**. The channels are then
sorted from highest to lowest.

**Decibels** (dB) put power on a log scale: every 10 dB is 10 times the power. Here the reference is
1 fT²/Hz, so 30 dB means 1,000 fT²/Hz. (1 femtotesla, fT, is 10⁻¹⁵ tesla.)

The **highest** average power usually means the **noisiest** channel; the lowest, the quietest. This uses the
spectra from Step 8, so bad channels are not ranked.

In [ ]:
psds = psd_all.get_data()           # one spectrum per channel, in T²/Hz (rows: channels, columns: frequencies)
freqs = psd_all.freqs               # the frequencies (Hz)
psd_channels = psd_all.ch_names     # the channels, in the same order as the rows

# T²/Hz -> fT²/Hz (1 T² = 1e30 fT²) -> decibels
psds_db = 10 * np.log10(psds * 1e30)
average_power = psds_db.mean(axis=1)  # the average over frequencies: one number per channel

ranking = pd.DataFrame({"channel": psd_channels,
                        "sensor": [sensor_of(n) for n in psd_channels],
                        "direction": [direction_of(n) for n in psd_channels],
                        "average_power_dB": average_power})
ranking = ranking.sort_values("average_power_dB", ascending=False).reset_index(drop=True)
ranking.index = ranking.index + 1     # rank 1 = the highest power
ranking.index.name = "rank"

# The row number of each channel in psds_db, in ranked order (to draw the spectra in ranked order below)
rows_ranked = [psd_channels.index(name) for name in ranking["channel"]]

print(f"{len(ranking)} channels ranked. Highest average power (noisiest):")
display(ranking.head(10).round(1))
print("Lowest average power (quietest):")
display(ranking.tail(10).round(1))

### The spectrum of every channel, in ranked order

One small plot per channel, from rank 1 (highest power) to the last, each with the **average spectrum over all
channels in red** for comparison. All plots use the **same scale**, so you can compare them directly. A
channel whose blue line sits well above the red one along the whole frequency range is a noisy channel; one
that sits above it only in part of the range may have a specific problem (for example a peak at one frequency).

With about 190 channels this is a tall figure: scroll through it.

In [ ]:
n_cols = 6                                           # plots per row
n_rows = int(np.ceil(len(rows_ranked) / n_cols))     # enough rows for every channel
grand_average = psds_db.mean(axis=0)                 # the average spectrum over all channels (red line)
y_low = np.percentile(psds_db, 0.5) - 3              # one shared y-range for all plots, with a little room
y_high = np.percentile(psds_db, 99.9) + 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 2.2 * n_rows), sharex=True, sharey=True)
axes = axes.flatten()                                # a simple list of the plots, row by row
for rank, row in enumerate(rows_ranked, start=1):
    ax = axes[rank - 1]
    ax.plot(freqs, grand_average, color="red", lw=0.8)       # the average over all channels
    ax.plot(freqs, psds_db[row], color="tab:blue", lw=0.8)   # this channel
    ax.set_title(f"{rank}. {psd_channels[row]}  ({average_power[row]:.1f} dB)", fontsize=9)
    ax.set_ylim(y_low, y_high)
    ax.grid(True, lw=0.3)
for ax in axes[len(rows_ranked):]:                   # hide the empty plots at the end
    ax.axis("off")
for ax in axes[(n_rows - 1) * n_cols:]:              # label only the bottom row and the left column
    ax.set_xlabel("Frequency (Hz)")
for ax in axes[::n_cols]:
    ax.set_ylabel("dB [fT²/Hz]")
fig.suptitle("Spectra ranked by average power (blue: channel, red: average of all channels)", y=1.0)
fig.tight_layout()
plt.show()

## Step 10: A bar chart of the average power

The same ranking as one bar per channel, highest on the left, coloured by direction. This shows at a glance
whether a few channels stand out, or whether the power falls off gradually. The dashed line is the median over
all channels.

The channels are split over two charts so the names stay readable.

In [ ]:
colours = {"X": "tab:orange", "Y": "tab:green", "Z": "tab:blue", "?": "tab:gray"}
median_power = ranking["average_power_dB"].median()
y_low = ranking["average_power_dB"].min() - 5
y_high = ranking["average_power_dB"].max() + 5
half = int(np.ceil(len(ranking) / 2))                 # two charts when there are many channels
parts = [ranking.iloc[:half], ranking.iloc[half:]] if len(ranking) > 40 else [ranking]

for i, part in enumerate(parts, start=1):
    fig, ax = plt.subplots(figsize=(18, 4.5))
    ax.bar(part["channel"], part["average_power_dB"], color=[colours[d] for d in part["direction"]])
    ax.axhline(median_power, color="k", ls="--", lw=1)
    ax.set_ylim(y_low, y_high)
    ax.set_ylabel("Average power, dB [fT²/Hz]")
    ax.set_title(f"OPM channels ranked by average power ({PSD_FMIN:g}-{fmax:g} Hz), part {i} of {len(parts)}")
    ax.tick_params(axis="x", labelrotation=90, labelsize=7)
    ax.margins(x=0.005)
    # A legend for the colours: one coloured square per direction
    ax.legend(handles=[Patch(color=colours[d], label=d) for d in ("X", "Y", "Z")],
              title="Direction", loc="upper right")
    fig.tight_layout()
    plt.show()

## Step 11: Time courses of the noisiest and quietest channels

The spectra summarise; the **time course** shows what the noise looks like: steady, in bursts, or a slow drift.
This plots the 3 channels ranked highest (**red**) and the 3 ranked lowest (**blue**) in Step 9, over the
**whole recording**.

Each channel has its mean subtracted, so they are centred on 0, and gets its own row. All rows share **one
y-scale**, so their sizes compare directly. Values are in **picotesla** (pT; 1 pT = 1,000 fT), and each row's
label gives its **standard deviation** (SD), a single number for how much it varies.

Over the whole recording, what you mostly see is **slow drift**: OPMs record the slow changes of the
background field (and of the head's position in it) that a conventional MEG system filters out. That is
normal, and later preprocessing removes it. A channel whose drift is much larger than the others, or that
jumps suddenly, is worth noting.

The function defined in this cell is used again in Steps 12 and 13.

In [ ]:
def plot_traces(names, colours, tmin=None, tmax=None, same_scale=True, remove_trend=False, title=""):
    # One row per channel in `names`, coloured by `colours` (a list, one colour per channel), from tmin to
    # tmax seconds (None: from the start / to the end). Each trace has its mean subtracted; with
    # remove_trend=True, a straight-line trend instead (Step 12 explains why). same_scale=True gives every
    # row the same y-range, so the rows can be compared; False lets each row fill its own.
    data, times = raw.get_data(picks=names, tmin=tmin or 0.0, tmax=tmax, return_times=True)
    if remove_trend:
        data = detrend(data, axis=1, type="linear")                  # straight-line trend removed
    else:
        data = data - data.mean(axis=1, keepdims=True)               # mean removed
    data = data * 1e12                                               # tesla -> picotesla
    fig, axes = plt.subplots(len(names), 1, figsize=(14, 1.3 * len(names) + 1.0),
                             sharex=True, sharey=same_scale, squeeze=False)
    for ax, name, colour, trace in zip(axes[:, 0], names, colours, data):
        ax.plot(times, trace, color=colour, lw=0.4, rasterized=True)   # rasterized: quicker to draw
        bad_note = "  (bad)" if name in raw.info["bads"] else ""
        # The row's label: channel name, and how much the plotted signal varies (its standard deviation)
        ax.set_ylabel(f"{name}{bad_note}\nSD {trace.std():.3g} pT", rotation=0, ha="right", va="center",
                      fontsize=9)
        ax.grid(True, lw=0.3)
    axes[-1, 0].set_xlabel("Time (s)")
    scale_note = "all rows on the same scale" if same_scale else "each row on its own scale"
    trend_note = "straight-line trend removed" if remove_trend else "mean removed"
    axes[0, 0].set_title(f"{title}\n({scale_note}, pT; {trend_note})")
    fig.tight_layout()
    plt.show()


noisiest = list(ranking["channel"].head(N_RANKED))   # rank 1, 2, 3
quietest = list(ranking["channel"].tail(N_RANKED))   # the last three
print(f"{N_RANKED} noisiest channels (highest power):", noisiest)
print(f"{N_RANKED} quietest channels (lowest power):", quietest)

plot_traces(noisiest + quietest, ["red"] * len(noisiest) + ["tab:blue"] * len(quietest),
            title=f"{N_RANKED} noisiest (red) and {N_RANKED} quietest (blue) channels, whole recording")

### The channels flagged in Step 7

The flat and noisy channels the screen found, also over the whole recording (at most `MAX_FLAGGED_PLOT`,
the most extreme first). Flat channels are drawn in **grey**, noisy ones in **red**. Here each row has **its own scale**: a flat channel on a shared scale would be a
straight line next to a noisy one, which tells you nothing about what little signal it has. Look at the numbers
on each row's axis to see how big its signal is.

In [ ]:
# The flagged channels, most extreme first (the screen table is already sorted that way)
flagged_table = screen[screen["flag"] != ""].head(MAX_FLAGGED_PLOT)
flagged_names = list(flagged_table["channel"])
flagged_colours = ["gray" if flag == "flat" else "red" for flag in flagged_table["flag"]]
n_flagged = len(flat_channels) + len(noisy_channels)
if n_flagged > len(flagged_names):
    print(f"Step 7 flagged {n_flagged} channels; plotting the {len(flagged_names)} most extreme "
          "(MAX_FLAGGED_PLOT in Step 1). So many usually means the limit is too strict for this "
          "recording, or a problem affecting many sensors at once (e.g. movement): check the Step 8 spectra.")

if not flagged_names:
    print("Step 7 found no flat or noisy channels: nothing to plot here.")
else:
    plot_traces(flagged_names, flagged_colours, same_scale=False,
                title="Channels flagged in Step 7: flat (grey) and noisy (red), whole recording")

## Step 12: A closer look: 5 seconds of the noisiest and quietest channels

Over the whole recording, slow drifts and big artifacts dominate. A short stretch (`STRETCH_DUR_S`, 5 s,
starting at `STRETCH_START_S`; by default the middle of the recording) shows the fast part of the signal: line
noise shows as a regular 60 Hz ripple (60 cycles per second, so very dense), heartbeats as spikes about once a
second, and brain rhythms like alpha as waves about 10 times a second.

Even over 5 seconds, the slow drift can tilt a trace so much that the fast part is squashed flat. So here each
trace has its **straight-line trend** removed (the best-fitting straight line through the 5 seconds is
subtracted), not just its mean. Nothing else is filtered, and the data in memory are not changed.

In [ ]:
# The stretch to show: STRETCH_START_S, or the middle of the recording when it is None
duration = raw.times[-1]
stretch_start = (duration - STRETCH_DUR_S) / 2 if STRETCH_START_S is None else STRETCH_START_S
stretch_start = min(max(stretch_start, 0.0), max(duration - STRETCH_DUR_S, 0.0))   # keep it inside the recording
stretch_end = stretch_start + STRETCH_DUR_S
print(f"Showing {stretch_start:.1f}-{stretch_end:.1f} s of a {duration:.1f} s recording")

plot_traces(noisiest + quietest, ["red"] * len(noisiest) + ["tab:blue"] * len(quietest),
            tmin=stretch_start, tmax=stretch_end, remove_trend=True,
            title=f"{N_RANKED} noisiest (red) and {N_RANKED} quietest (blue) channels, "
                  f"{stretch_start:.1f}-{stretch_end:.1f} s")

## Step 13: 5 seconds of the flagged channels

The same stretch for the flat (grey) and noisy (red) channels from Step 7, each on its own scale, with the
straight-line trend removed as in Step 12. For a flat channel, compare its SD with the SDs in Step 12: a dead
channel varies thousands of times less than a working one.

In [ ]:
if not flagged_names:
    print("Step 7 found no flat or noisy channels: nothing to plot here.")
else:
    plot_traces(flagged_names, flagged_colours, tmin=stretch_start, tmax=stretch_end, same_scale=False,
                remove_trend=True,
                title=f"Channels flagged in Step 7: flat (grey) and noisy (red), "
                      f"{stretch_start:.1f}-{stretch_end:.1f} s")

## Step 14: Inspect the data with the interactive browser

`raw.plot()` is MNE's data browser. It shows the **Z-direction** channels, `BROWSE_DURATION_S` seconds at a
time, starting at `BROWSE_START_S`. Bad channels are drawn in grey. The browser removes each channel's mean, so
the large constant offsets of the OPMs don't push the traces off the screen, and by default it also applies a
1 Hz high-pass filter to **what it draws** (`BROWSE_HIGHPASS_HZ`), so the slow drift doesn't tilt every trace.
The data themselves are not filtered.

`%matplotlib widget` (from `ipympl`) makes the window interactive inside the notebook, both on scotty and on
your laptop. Click inside the figure first so it has keyboard focus, then:

- **← / →** scroll in time
- **↑ / ↓** scroll through channels
- **Click a trace** to mark that channel bad, or click it again to unmark it. The change goes into
  `raw.info["bads"]` (in memory) when you close the figure with the × in its toolbar. This is **not**
  saved: to keep it, add the channel to `BAD_CHANNELS` (with a reason in `BAD_REASONS`) in Step 7b and run
  from there again, which also re-saves the list (Step 7c).
- **?** shows the full list of keys.

On scotty, every redraw travels over the tunnel, so keep `BROWSE_DURATION_S` small if it feels sluggish.

Things to look for while you scroll:

- **Eye blinks**: large, slow deflections (a few hundred ms) on the front sensors. Notebook 05 marked the
  blinks the eye tracker saw as `ET_blink` annotations, and the stretches where it lost the eye as
  `ET_missing`, drawn as shaded spans. The task events from `_events.tsv` (`standard`, `deviant`,
  `left cue`, `right cue`, `blink`) are shaded too.
- **Heartbeats**: small, regular spikes about once a second, often clearest on the lower sensors.
- **Muscle activity**: bursts of fast, fuzzy activity, often on the temporal sensors (jaw clenching).
- **Movement**: large slow shifts on many channels at once. OPMs are sensitive to head movement in the
  background field.

In [ ]:
# Switch to interactive plots (needs ipympl) for the browser
%matplotlib widget

mne.viz.set_browser_backend("matplotlib")   # draw the browser with matplotlib, so it works in the notebook
# highpass= filters only what is drawn (the data in memory stay as they are)
raw.plot(picks=OPMZ, start=BROWSE_START_S, duration=BROWSE_DURATION_S, n_channels=30,
         scalings=dict(mag=2e-11), highpass=BROWSE_HIGHPASS_HZ, title=f"{base}: Z channels", show=True);

## Next: loading this session in the next notebook

The bad channels are **not** written into the recording. The recording stays where notebook 05 put it, and
this notebook only adds the list of bad channels next to it, in its own folder:

| What | Where |
|---|---|
| The eye-synced recording (MEG + eye channels) and its sidecar files | `derivatives/eyesync/sub-<label>/ses-<label>/meg/…_proc-eyesync_…` |
| The bad channels, with the reason for each (Step 7c) | `derivatives/sensorqc/sub-<label>/ses-<label>/meg/…_proc-eyesync_channels.tsv` |

So the next notebook loads the recording **the same way Step 2 did**, then takes the bad channels from
sensorqc. Each piece links back to the other: sensorqc's `dataset_description.json` lists the eyesync folder
under `DatasetLinks`, and its `_channels.json` names the exact FIF file(s) the list belongs to under `Sources`.

```python
import mne, mne_bids, pandas as pd

# 1. The eye-synced recording, from derivatives/eyesync (as in Step 2)
bids_path = mne_bids.BIDSPath(root=BIDS_ROOT / "derivatives" / "eyesync", subject=SUBJECT, session=SESSION,
                              task=TASK, run=RUN, processing="eyesync", split="01",   # split=None if not split
                              datatype="meg", suffix="meg", extension=".fif")
raw = mne_bids.read_raw_bids(bids_path)      # warns "participants.tsv file not found": expected (see Step 2)
# ... then the repair of the eye-tracker channels from Step 2, copied as is ...

# 2. The bad channels, from derivatives/sensorqc
sensorqc_tsv = (BIDS_ROOT / "derivatives" / "sensorqc" / f"sub-{SUBJECT}" / f"ses-{SESSION}" / "meg"
                / bids_path.copy().update(split=None, suffix="channels", extension=".tsv").basename)
channels = pd.read_csv(sensorqc_tsv, sep="\t")
raw.info["bads"] = list(channels.loc[channels["status"] == "bad", "name"])
```

Two things to keep in mind:

- **Use this notebook's list, not the one in `derivatives/eyesync/`.** `read_raw_bids` already sets
  `raw.info["bads"]` from the eyesync `_channels.tsv`, but that file only has the channels marked bad *before*
  this notebook. Step 2 above replaces that list with the one from sensorqc.
- **If you change the bad channels later**, change them here (Step 7b) and re-save (Step 7c), so there is
  always one up-to-date list.

The first later notebook that changes the data itself (for example filtering or removing the background field)
will save a complete new recording, with these bad channels in it, in its own derivatives folder. From then on,
each notebook reads the output of the one before it.